# Deep Learning Forum 04: Indonesian Food Classification with CNN

Notebook ini membangun pipeline end-to-end untuk klasifikasi gambar makanan Indonesia menggunakan Convolutional Neural Network (CNN). Alur kerja mencakup akuisisi dataset dari Kaggle, pelabelan data berdasarkan struktur folder, exploratory data analysis (EDA), preprocessing gambar ke ukuran `224 x 224`, training model CNN kustom, dan perbandingan dengan model transfer learning berbasis VGG16.

Fokus evaluasi utama pada tugas ini adalah:
1. konsistensi pipeline data gambar,
2. kualitas generalisasi model pada validation dan test split,
3. perbandingan antara arsitektur CNN sederhana versus transfer learning.

In [ ]:
import os

# Inisiasi token cukup sekali di cell ini.
# Isi token Anda secara lokal saat runtime, lalu jangan simpan nilai token pada notebook.
KAGGLE_API_TOKEN = 'KGAT_d196137f2c449462d638078966bba79b'  # Kosongkan di repository untuk keamanan.

if KAGGLE_API_TOKEN.strip():
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN.strip()
else:
    # Fallback ke environment sistem jika sudah diset dari terminal/OS.
    os.environ['KAGGLE_API_TOKEN'] = os.getenv('KAGGLE_API_TOKEN', '').strip()

In [ ]:
from typing import List, Dict

In [10]:
import os
import warnings
import tensorflow as tf

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# TensorFlow >= 2.11 on native Windows does not support GPU.
if os.name == "nt":
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")

# List available hardware
if os.name == "nt":
    devices: List = tf.config.list_physical_devices("CPU")
else:
    devices = tf.config.list_physical_devices()

print(f"Devices found: {devices}")

Devices found: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
if os.name == "nt":
    print("Running on Local CPU (native Windows TensorFlow build)")
else:
    gpu_devices: List = tf.config.list_physical_devices("GPU")

    if gpu_devices:
        print(f"GPU devices found: {gpu_devices}")

        for each_gpu_device in gpu_devices:
            tf.config.experimental.set_memory_growth(each_gpu_device, True)

    else:
        print("Running on Local CPU")

## Remarks: How to Fetch the Dataset

We will use `Indonesian Food Dataset` from [Kaggle](https://www.kaggle.com/datasets/rizkyyk/dataset-food-classification) for this project.

### Dataset Description

The dataset contains food images grouped into class folders and split into training, validation, and test subsets. The primary categories used in this project are:
* Ayam Goreng
* Burger
* French Fries
* Gado-Gado
* Ikan Goreng
* Mie Goreng
* Nasi Goreng
* Nasi Padang
* Pizza
* Rawon
* Rendang
* Sate
* Soto Ayam

### Authentication

Authentication is needed to download this dataset from Kaggle.

First, create a Kaggle account at [Kaggle Login](https://www.kaggle.com/account/login). After logging in, generate your API token from [Kaggle Settings](https://www.kaggle.com/settings) under the `API` section.

You have several options to authenticate:

Option 1: `kagglehub.login()`

```python
import kagglehub

kagglehub.login()
```

Option 2: Environment Variable

```bash
export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx
```

Option 3: API Token File

Store your Kaggle API token in `~/.kaggle/access_token`.

Option 4: Google Colab Secret

If you run this notebook on Google Colab, store the token as a secret named `KAGGLE_API_TOKEN`.

Option 5: Legacy API credentials file

Download `kaggle.json` from [Kaggle Settings](https://www.kaggle.com/settings) and store it at `~/.kaggle/kaggle.json`.

In [ ]:
# Authenticate Kaggle

import os
from pathlib import Path


import kagglehub

is_authenticated: bool = False

api_key_path: Path = Path("~/.kaggle/kaggle.json").expanduser()
is_api_key_exists: bool = True if api_key_path.exists() else False

api_token_path: Path = Path("~/.kaggle/access_token").expanduser()
is_api_token_exists: bool = True if api_token_path.exists() else False

## Check if running in Colab
def is_colab() -> bool:
    """
    Small utility function to check if this notebook is running on Colab
    :return: Boolean indication if this script is running on Colab
    :rtype bool
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False

is_colab_secret_exists: bool = False
if is_colab():
    from google.colab import userdata

    try:
        is_colab_secret_exists = True if userdata.get("KAGGLE_API_TOKEN") else False
    except userdata.SecretNotFoundError:
        is_colab_secret_exists = False

is_token_env_exists: bool = True if os.getenv("KAGGLE_API_TOKEN") else False

is_token_via_login: bool = False
if all([
    not is_api_key_exists
    , not is_api_token_exists
    , not is_colab_secret_exists
    , not is_token_env_exists
]):
    # if everything above fails, then login interactively
    # this will ask you to input the Kaggle API Token from a UI
    kagglehub.login()
    is_token_via_login = True


if any([
    is_api_key_exists
    , is_api_token_exists
    , is_token_env_exists
    , is_colab_secret_exists
    , is_token_via_login
]):
    is_authenticated = True
    print("Kagglehub has been authenticated")

else:
    raise Exception(" 🛑 Kagglehub fails to authenticate")

## Remarks: Download the Dataset

In this section, we will download the dataset from Kaggle.

We want to detect if any `.jpg` files already present under `dataset/` folder.

If not, then we: (1) would download the dataset from Kaggle, (2) copy the downloaded data from Kaggle's `cache/` folder into the project's `dataset/` folder. If `.jpg` files already present in the `dataset/` folder, then we will simply load the data from the `dataset/` folder.

This way, we can avoid downloading the dataset every time we run this notebook.

In [9]:
# Download data smartly
# If image files are already present under `dataset/`, we reuse them.
# Otherwise, we download the dataset from Kaggle once and copy it into the project dataset folder.

from pathlib import Path
import shutil

import kagglehub

def resolve_project_root() -> Path:
    current_dir = Path.cwd().resolve()
    candidate_dirs = [current_dir, current_dir.parent]
    required_markers = {
        "Forum04-indonesian_food_question.ipynb",
        "Forum04-requirements.txt",
    }

    for each_candidate in candidate_dirs:
        if all((each_candidate / each_marker).exists() for each_marker in required_markers):
            return each_candidate

    raise FileNotFoundError(
        "Forum04 project root was not found. Please run this notebook from the forum04 workspace folder."
    )

kaggle_dataset_handler_str: str = "rizkyyk/dataset-food-classification"

project_root: Path = resolve_project_root()
dataset_folder: Path = project_root / "dataset"
dataset_folder.mkdir(parents = True, exist_ok = True)

valid_image_extensions = {".jpg", ".jpeg", ".png"}

available_files = sorted(
    each_found_file.name
    for each_found_file in dataset_folder.glob("**/*")
    if each_found_file.is_file() and each_found_file.suffix.lower() in valid_image_extensions
)

print(f"Project root resolved to: {project_root}")
print(f"Dataset folder resolved to: {dataset_folder}")

if available_files:
    print("Dataset already exists locally. Using files from `dataset/` folder.")
    print(f"Count of available image files: {len(available_files)}")
else:
    downloaded_path_str: str = kagglehub.dataset_download(
        handle = kaggle_dataset_handler_str
    )
    downloaded_path: Path = Path(downloaded_path_str)

    candidate_folders = [
        downloaded_path / "dataset_gambar",
        downloaded_path / "dataset",
        downloaded_path,
    ]

    source_folder = next(
        (each_folder for each_folder in candidate_folders if each_folder.exists()),
        None,
    )

    if source_folder is None:
        raise FileNotFoundError("Unable to locate downloaded dataset folder structure.")

    shutil.copytree(
        src = source_folder,
        dst = dataset_folder,
        dirs_exist_ok = True,
    )

    available_files = sorted(
        each_found_file.name
        for each_found_file in dataset_folder.glob("**/*")
        if each_found_file.is_file() and each_found_file.suffix.lower() in valid_image_extensions
    )

    print(f"Count of available image files: {len(available_files)}")

Project root resolved to: D:\Project S2\2521\2521-deepLearning-forum04
Dataset folder resolved to: D:\Project S2\2521\2521-deepLearning-forum04\dataset
Resuming download from 123731968 bytes (1047403551 bytes left)...
Resuming download to C:\Users\yanli\.cache\kagglehub\datasets\rizkyyk\dataset-food-classification\5.archive (123731968/1171135519) bytes left.


100%|██████████| 1.09G/1.09G [04:09<00:00, 4.21MB/s]

Extracting files...


Count of available image files: 6490


## Remark: Data Pre-Processing - Extract Categories Name from Folder Name

The `indonesian_food` image dataset are grouped into their respective folders, e.g. 'Ayam Goreng', 'Burger', "French Fries', 'Mie Goreng', etc. As such to convert the images into their respective categories, we will need to extract the folder name as the category name.

In [ ]:
from pathlib import Path

train_folder: Path = dataset_folder / "train"

if not train_folder.exists():
    raise FileNotFoundError("`dataset/train` folder was not found. Please run the dataset download cell first.")

folder_names: List[str] = sorted(
    each_folder.name
    for each_folder in train_folder.iterdir()
    if each_folder.is_dir()
    )

print(folder_names)

In [ ]:
# Create small utility function to convert categories into machine friendly format
from typing import List, Dict
import re

def slugify(
    text: str
) -> str:

    """
    Converts a given string to a URL-friendly format (slugified version). This involves
    lowercasing the text, removing non-alphanumeric characters, and replacing
    whitespace with underscores.

    :param text: The input string to be converted.
    :type text: str
    :return: A slugified version of the input string.
    :rtype: str
    """

    my_text: str = text.lower()
    my_text = re.sub(
        pattern = r"[^\w\s]"
        , repl = ""
        , string = my_text
    )
    my_text = re.sub(
        pattern = r"\s+"
        , repl = "_"
        , string = my_text
    )

    return my_text


def list_to_slug_dict(
        list_items: List[str]
) -> Dict[str, str]:

    return {each_item: slugify(each_item) for each_item in list_items}

In [ ]:
from pprint import pprint

categories_name: Dict[str, str] = list_to_slug_dict(list_items = folder_names)

pprint(categories_name, indent = 4)

## Remark: Data Pre-Processing = Associate Each Train / Val / Test Images to Label

To do this, we will enlist each image paths from the `dataset/` folder, and associate the image with appropriate label derived from the parent folder where it comes from

In [ ]:
## Create small utility function that allows us to apply appropriate label from its parent folder
from typing import List, Dict, Union

from pathlib import Path

ListLabels = List[Dict[str, Union[str, Path]]]

def label_images(
    list_image_paths: List[Path]
    , categories_name: Dict[str, str]
) -> ListLabels:

    labeled_image: ListLabels = []

    for each_image_path in list_image_paths:

        image_parent_folder: Path = each_image_path.parent
        label_text = categories_name[image_parent_folder.name]

        each_labeled_image: Dict[str, Union[str, Path]] = {
            "image_path": each_image_path
            , "label": label_text
        }

        labeled_image.append(each_labeled_image)

    return labeled_image

In [ ]:
## Label Train, Validation, and Test Images

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def resolve_split_directory(
    base_folder: Path,
    candidates: List[str],
) -> Path:

    for each_candidate in candidates:
        candidate_path = base_folder / each_candidate
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(
        f"None of the split directories {candidates} were found under {base_folder}."
    )

def list_image_paths(
    image_folder: Path,
    image_extensions: set = IMAGE_EXTENSIONS,
    ) -> List[Path]:

    return sorted(
        each_file
        for each_file in image_folder.glob("**/*")
        if each_file.is_file() and each_file.suffix.lower() in image_extensions
    )

train_image_folder: Path = resolve_split_directory(dataset_folder, ["train"])
train_image_paths: List[Path] = list_image_paths(image_folder = train_image_folder)
labeled_train_images: ListLabels = label_images(
    list_image_paths = train_image_paths,
    categories_name = categories_name,
    )

val_image_folder: Path = resolve_split_directory(dataset_folder, ["valid", "val"])
val_image_paths: List[Path] = list_image_paths(image_folder = val_image_folder)
labeled_val_images: ListLabels = label_images(
    list_image_paths = val_image_paths,
    categories_name = categories_name,
    )

test_image_folder: Path = resolve_split_directory(dataset_folder, ["test"])
test_image_paths: List[Path] = list_image_paths(image_folder = test_image_folder)
labeled_test_images: ListLabels = label_images(
    list_image_paths = test_image_paths,
    categories_name = categories_name,
    )

print(f"Train images: {len(labeled_train_images)}")
print(f"Validation images: {len(labeled_val_images)}")
print(f"Test images: {len(labeled_test_images)}")

## Remark: Data Pre-Processing - Exploratory Data Analysis

In the EDA Section, we will explore few things:

* we will explore the distribution of image count for each categories. We want to do this, because CNN weight training can be easily dominated by imbalance class
* we will explore the image size distribution. We want to do this to see if we need to introduce image resizing if there are image size variation

## Utility Function to Visualize Label Distribution

This utility function compares the image count for each label across the train, validation, and test sets.

In [ ]:
from collections import Counter
from typing import Dict

import matplotlib.pyplot as plt

def visualize_label_distribution(
    labeled_train_images: ListLabels
    , labeled_val_images: ListLabels
    , labeled_test_images: ListLabels
) -> Dict[str, Dict[str, int]]:

    split_to_images: Dict[str, ListLabels] = {
        "train": labeled_train_images
        , "val": labeled_val_images
        , "test": labeled_test_images
    }

    ordered_labels = sorted({
        str(each_image["label"])
        for each_split in split_to_images.values()
        for each_image in each_split
    })

    fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 6), sharey = True)
    split_counts: Dict[str, Dict[str, int]] = {}
    colors = {
        "train": "#2E86AB"
        , "val": "#F18F01"
        , "test": "#C73E1D"
    }

    for each_axis, (split_name, labeled_images) in zip(axes, split_to_images.items()):

        label_counter = Counter(str(each_image["label"]) for each_image in labeled_images)
        label_counts = {
            each_label: label_counter.get(each_label, 0)
            for each_label in ordered_labels
        }
        split_counts[split_name] = label_counts

        each_axis.bar(
            x = list(label_counts.keys())
            , height = list(label_counts.values())
            , color = colors[split_name]
        )
        each_axis.set_title(f"{split_name.title()} Label Distribution")
        each_axis.set_xlabel("Label")
        each_axis.tick_params(axis = "x", rotation = 90)

    axes[0].set_ylabel("Image Count")
    fig.suptitle("Label Count Distribution Across Dataset Splits")
    plt.tight_layout()
    plt.show()

    return split_counts


In [ ]:
label_distribution = visualize_label_distribution(
    labeled_train_images = labeled_train_images,
    labeled_val_images = labeled_val_images,
    labeled_test_images = labeled_test_images,
    )

label_distribution

## Utility Function to Visualize Image Size Distribution

This utility function reads each image from the train, validation, and test sets, then compares the width and height distributions across all splits.

In [ ]:
from typing import Any

from PIL import Image

def visualize_image_size_distribution(
    labeled_train_images: ListLabels
    , labeled_val_images: ListLabels
    , labeled_test_images: ListLabels
) -> Dict[str, Dict[str, List[Any]]]:

    split_to_images: Dict[str, ListLabels] = {
        "train": labeled_train_images
        , "val": labeled_val_images
        , "test": labeled_test_images
    }

    size_summary: Dict[str, Dict[str, List[Any]]] = {}
    fig, axes = plt.subplots(nrows = 2, ncols = 3, figsize = (20, 10), sharey = "row")
    colors = {
        "train": "#2E86AB"
        , "val": "#F18F01"
        , "test": "#C73E1D"
    }

    for each_column, (split_name, labeled_images) in enumerate(split_to_images.items()):

        widths: List[int] = []
        heights: List[int] = []
        image_sizes: List[tuple] = []

        for each_image in labeled_images:
            image_path = each_image["image_path"]

            with Image.open(image_path) as img:
                width, height = img.size

            widths.append(width)
            heights.append(height)
            image_sizes.append((width, height))

        size_summary[split_name] = {
            "widths": widths
            , "heights": heights
            , "sizes": image_sizes
        }

        axes[0, each_column].hist(widths, bins = 20, color = colors[split_name], alpha = 0.85)
        axes[0, each_column].set_title(f"{split_name.title()} Width Distribution")
        axes[0, each_column].set_xlabel("Width (pixels)")

        axes[1, each_column].hist(heights, bins = 20, color = colors[split_name], alpha = 0.85)
        axes[1, each_column].set_title(f"{split_name.title()} Height Distribution")
        axes[1, each_column].set_xlabel("Height (pixels)")

    axes[0, 0].set_ylabel("Image Count")
    axes[1, 0].set_ylabel("Image Count")
    fig.suptitle("Image Size Distribution Across Dataset Splits")
    plt.tight_layout()
    plt.show()

    return size_summary


In [ ]:
image_size_distribution = visualize_image_size_distribution(
    labeled_train_images = labeled_train_images,
    labeled_val_images = labeled_val_images,
    labeled_test_images = labeled_test_images,
    )

{
    each_split: {
        "min_width": min(each_summary["widths"]),
        "max_width": max(each_summary["widths"]),
        "min_height": min(each_summary["heights"]),
        "max_height": max(each_summary["heights"]),
    }
    for each_split, each_summary in image_size_distribution.items()
}

From the result above, we notice that the image size are not all equal

Instead, the images have distribution of image size - which will complicate the CNN model downstream.

Hence, before we can proceed with creating a CNN model, we need to introduce another utility function to help us standardizing the image size

## Remarks: Data Pre-Processing - Transform Image Dataset into `ImageNet` image size (224, 224, 3)

As noted earlier, our `indonesian_food` images have various image sizes.

Before we can create our CNN model, we want to uniformize the image size so that the model would accept only a single array size.

For this task, we would pick the image size found in `ImageNet` dataset, namely: (224, 224, 3)

Why we pick `ImageNet` image size? We pick it, because `ImageNet` is standard image dataset that has been used for training other CNN family models. Hence, by transforming our `indonesian_food` images into `ImageNet` image size, we can later play around with several CNN model families

To do this transformation, we will do three steps:
1. Create a function to scale the shortest dimension to `224`. For example, if the original image has a size of (800, 600) , we want to convert the size into (300, 224). This way, we would keep the aspect ratio (H : W) of the image constant
2. Crop the resized output from (1) from the center so that it produce a new image with size (224, 224, 3)
3. Wrap (1) and (2) into a single function

## 🧠Food For Thought

The preprocessing pipeline resizes each image and then takes a center crop of size `224 x 224`. Discuss how this choice could help or hurt classification performance for food images. In your answer, think about cases where the main dish is not centered or where important visual details appear near the edges of the image.


In [ ]:
## Define function to resize the smallest dimension to 224 pixel

from PIL import Image
import numpy as np

def resize_shortest(
        img: Image.Image
) -> np.ndarray:

    """Resize an image so its shortest side becomes 224 pixels.

    This function converts the input image to RGB mode, computes a scale
    factor from the shortest spatial dimension, and resizes the image while
    preserving its aspect ratio. The resized image is returned as a NumPy
    array with three RGB channels.

    Parameters
    ----------
    img : PIL.Image.Image
        Input image to be resized.
        Default value: no default value.

    Returns
    -------
    np.ndarray
        Resized RGB image as a NumPy array with shape `(height, width, 3)`.
    """

    TARGET_SIZE: int = 224

    # Ensure the image is in RGB format
    rgb_image: Image.Image = img.convert("RGB")

    scale_factor: float = TARGET_SIZE / min(rgb_image.height, rgb_image.width)
    new_w: int = round(rgb_image.width * scale_factor)
    new_h: int = round(rgb_image.height * scale_factor)

    resized_image: Image.Image = rgb_image.resize(
        size = (new_w, new_h)
        , resample = Image.Resampling.LANCZOS
    )

    scaled_image_np: np.ndarray = np.array(resized_image)

    return scaled_image_np


In [ ]:
# Define function to crop image from the center
def crop_from_center(
        img: np.ndarray
) -> np.ndarray:

    """Crop the center region of an image to 224 x 224 pixels.

    This function extracts a square crop from the center of a NumPy image
    array. It assumes the input image has already been resized so that both
    spatial dimensions are at least 224 pixels. The output preserves the
    original channel dimension and returns the centered crop as a NumPy
    array.

    Parameters
    ----------
    img : np.ndarray
        Input image as a NumPy array with shape `(height, width, channels)`.
        Default value: no default value.

    Returns
    -------
    np.ndarray
        Center-cropped image as a NumPy array with shape `(224, 224, channels)`.
    """

    TARGET_SIZE: int = 224

    image_h: int = img.shape[0]
    image_w: int = img.shape[1]

    if image_h < TARGET_SIZE or image_w < TARGET_SIZE:
        raise ValueError("Input image must be at least 224x224 before center cropping.")

    left_pt: int = (image_w - TARGET_SIZE) // 2
    right_pt: int = left_pt + TARGET_SIZE

    top_pt: int = (image_h - TARGET_SIZE) // 2
    bottom_pt: int = top_pt + TARGET_SIZE

    cropped_image_np: np.ndarray = img[top_pt:bottom_pt, left_pt:right_pt, :]

    return cropped_image_np

In [ ]:
# Encapsulate both resize and crop functions into a single function

def resize_and_crop_image(
        img: Image.Image
)-> np.ndarray:

    """Resize and Crop Image to (224, 224, 3) size

    This function takes an image as input and perform the following steps:
    1. Resize the image to (height, 224) or (224, height) using the resize_shortest function
    2. Crop the resized image to (224, 224, 3) using the crop_from_center function

    Parameters
    ---------
    :param img: PIL.Image.Image
        PIL Image object  to be resized and cropped
        Default value: no default value.

    Returns
    :return: np.ndarray
        Resized and Cropped Image as a NumPy array with shape (224, 224, 3)
    """

    # first, we resize the image
    resized_img: np.ndarray = resize_shortest(img = img)

    # Seccond, we crop the image from the center
    cropped_img: np.ndarray = crop_from_center(img = resized_img)

    return cropped_img

In [ ]:
# Test the resize-and-crop utiliy function

test_image_dict: Dict[str, Union[str, Path]] = labeled_train_images[0]
test_image_path: Path = test_image_dict["image_path"]
test_image: Image.Image = Image.open(test_image_path)

print(f"Original Image Size: {test_image.size}")
display(test_image)

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

prc_img: np.ndarray = resize_and_crop_image(img = test_image)

print(f"Resized and Cropped Image Size: {prc_img.shape}")
plt.imshow(prc_img)
plt.axis("off")
plt.show()

After we're done preparing the utility function to resize_and_crop images, we then proceed to create another set of utility functions to preprocess the images for training and inference.

These functions will be used to load and preprocess images in a TensorFlow dataset pipeline.

In [ ]:
from typing import Tuple, List, Dict, Union
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image

In [ ]:
# Convert string label names into integer labels
label_names: List[str] = sorted(categories_name.values())
label_to_id: Dict[str, int] = {
    each_label : idx for idx, each_label in enumerate(label_names)
}

In [ ]:
def prepare_records(
        labeled_images: ListLabels
) -> Tuple[List[str], List[int]]:

     image_paths: List[str] = [
         str(each_item["image_path"]) for each_item in labeled_images
     ]

     labels_int: List[int] = [
        label_to_id[each_item["label"]] for each_item in labeled_images
     ]

     return image_paths, labels_int

In [ ]:
def load_and_preprocess_image_py(
        image_path_tensor: tf.Tensor
) -> tf.Tensor:

    image_path: str = image_path_tensor.numpy().decode("utf-8")

    with Image.open(image_path) as img:
        rc_image_np: np.ndarray = resize_and_crop_image(img = img)

    rc_image_np = rc_image_np.astype(np.float32) / 255.0

    rc_image_tf: tf.Tensor = tf.convert_to_tensor(rc_image_np)

    return rc_image_tf

In [ ]:
def tf_preprocess_wrapper(
        image_path: tf.Tensor
        , label: tf.Tensor
) -> Tuple[tf.Tensor, tf.Tensor]:

    image: tf.Tensor = tf.py_function(
        func = load_and_preprocess_image_py
        , inp = [image_path]
        , Tout = tf.float32
    )

    image.set_shape((224,224,3))

    return image, label

In [ ]:
def build_dataset_from_labeled_images(
        labeled_images: ListLabels
        , batch_size: int = 32
        , is_training: bool = False
) -> tf.data.Dataset:

    image_paths, labels = prepare_records(labeled_images = labeled_images)

    dst: tf.data.Dataset = tf.data.Dataset.from_tensor_slices(
        tensors = (image_paths, labels)
    )

    if is_training:
        dst = dst.shuffle(buffer_size = len(image_paths))

    dst = dst.map(tf_preprocess_wrapper, num_parallel_calls = tf.data.AUTOTUNE)
    dst = dst.batch(batch_size = batch_size)
    dst = dst.prefetch(buffer_size = tf.data.AUTOTUNE)

    return dst

## Remark: Create CNN Model

Now that we have uniformize the image size, we can proceed to create a Convolutional Neural Network (CNN) model for image classification.

For this task, we will try-out with several CNN architectures. First, we will start with our CNN architecture, later will also try the same classification task with VGGNet and we can compare the performance of both models.

For this classification task, we will use accuracy as the evaluation metric.

## Model 01: Our CNN Architecture

For our own architecture, we will start using something simple:
* We will repeat a block of (CNN + Pooling Layer + Activation Layer) for several n-round
* We will add a simple MLP head to convert the features extracted from the CNN into appropriate class

In [ ]:
from typing import Tuple
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_cnn_block(
    x: tf.Tensor
    , filter: int
    , kernel_size: Tuple[int, int] = (3, 3)
    , pool_size: Tuple[int, int] = (2, 2)
    , activation: str = "relu"
) -> tf.Tensor:

    x = layers.Conv2D(filters = filter, kernel_size = kernel_size, padding = "same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    x = layers.Conv2D(filters = filter, kernel_size = kernel_size, padding = "same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    x = layers.MaxPooling2D(pool_size = pool_size)(x)

    return x


def build_model(
    input_shape: Tuple[int, int, int] = (224, 224, 3)
    , num_classes: int = len(categories_name)
    , conv_filters: Tuple[int] = (32, 64, 128, 256)
    , mlp_units : Tuple[int, int] = (128, 64)
    , activation: str = "relu"
) -> Model:

    inputs: tf.Tensor = layers.Input(shape = input_shape)
    x: tf.Tensor = inputs

    # Create repeated CNN blocks
    for each_filter in conv_filters:
        x = build_cnn_block(x = x, filter = each_filter, activation = activation)


    # Transition to MLP
    x = layers.GlobalAveragePooling2D()(x)

    # MLP head
    for each_layer in mlp_units:
        x = layers.Dense(units = each_layer)(x)
        x = layers.Activation(activation)(x)
        x = layers.Dropout(rate = 0.3)(x)

    outputs: tf.Tensor = layers.Dense(units = num_classes, activation = "softmax")(x)

    model: Model = Model(inputs = inputs, outputs = outputs)

    return model

In [ ]:
cnn_model: Model = build_model()
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4)
    , loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = False)
    , metrics = [
        "accuracy"
        , tf.keras.metrics.SparseTopKCategoricalAccuracy(k = 3, name = "top_3_accuracy")
    ]
)

Now that we have defined the architecture of the model, and we have also compiled the model, we can proceed with building the dataset by taking advantage of the dataset preparation function we have worked on in earlier section

In [ ]:
# Create folder to save the best model
from pathlib import Path

cwd: Path = Path.cwd()
model_path: Path = cwd / "keras_model"

if not model_path.exists():
    model_path.mkdir(parents = True, exist_ok = True)

In [ ]:
import tensorflow as tf

BATCH_SIZE: int = 32
EPOCHS: int = 30

train_ds: tf.data.Dataset = build_dataset_from_labeled_images(
    labeled_images = labeled_train_images
    , batch_size = BATCH_SIZE
    , is_training = True
)

val_ds: tf.data.Dataset = build_dataset_from_labeled_images(
    labeled_images = labeled_val_images
    , batch_size = BATCH_SIZE
    , is_training = False
)

test_ds: tf.data.Dataset = build_dataset_from_labeled_images(
    labeled_images = labeled_test_images
    , batch_size = BATCH_SIZE
    , is_training = False
)

In [ ]:
callbacks: List[tf.keras.callbacks.Callback] = [
    tf.keras.callbacks.EarlyStopping(
        monitor = "val_loss"
        , patience = 5
        , restore_best_weights = True
    )
    , tf.keras.callbacks.ReduceLROnPlateau(
        monitor = "val_loss"
        , factor = 0.5
        , patience = 2
        , min_lr = 1e-6
    )
    , tf.keras.callbacks.ModelCheckpoint(
        filepath = str(model_path / "indonesian_food_model.keras")
        , monitor = "val_loss"
        , save_best_only = True
    )
]

In [ ]:
# Train Model
history: tf.keras.callbacks.History = cnn_model.fit(
    train_ds
    , validation_data = val_ds
    , epochs = EPOCHS
    , callbacks = callbacks
)

In [ ]:
def plot_loss_history(
    history: tf.keras.callbacks.History
) -> None:

    """Plot training history across loss and accuracy metrics.

    Parameters
    ----------
    history : tf.keras.callbacks.History
        Keras History object returned by `model.fit()`.
        Default value: no default value.

    Returns
    -------
    None
        This function displays matplotlib figures and does not return a value.
    """

    history_dict = history.history
    epochs = range(1, len(history_dict.get("loss", [])) + 1)

    fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (18, 5))

    axes[0].plot(epochs, history_dict.get("loss", []), marker = "o", label = "Training Loss")
    axes[0].plot(epochs, history_dict.get("val_loss", []), marker = "o", label = "Validation Loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha = 0.3)

    axes[1].plot(epochs, history_dict.get("accuracy", []), marker = "o", label = "Training Accuracy")
    axes[1].plot(epochs, history_dict.get("val_accuracy", []), marker = "o", label = "Validation Accuracy")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha = 0.3)

    axes[2].plot(
        epochs,
        history_dict.get("top_3_accuracy", []),
        marker = "o",
        label = "Training Top-3 Accuracy",
    )
    axes[2].plot(
        epochs,
        history_dict.get("val_top_3_accuracy", []),
        marker = "o",
        label = "Validation Top-3 Accuracy",
    )
    axes[2].set_title("Top-3 Accuracy")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Top-3 Accuracy")
    axes[2].legend()
    axes[2].grid(alpha = 0.3)

    for each_axis in axes:
        each_axis.set_xticks(list(epochs))

    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss_history(history = history)


From the loss evolution graph above, it seems our training loss is worse than the validation loss. That is, the loss_training > loss_val in most of the epochs.

This is not neccessarily bad.

Some of the potential reason may include:
(1) the training data is noisier than than validation dataset. That is, the training dataset can be more difficult to learn than the validation dataset.
(2) the loss in training is computed while the weight in the models are still being optimized, while the loss in the validation is computed after each epoch, where the weight is no longer optimized for that moment. Hence, the difference can be caused by this differenece in measurement mode alone

To test which amongst these two possible explanations, we can run `cnn_model.evaluate()` for both `train_ds` and `val_ds` to see if the difference in loss values is still pronounced.

The idea is that if in the evaluation mode, the loss_training is still worse than loss_val, then the training image dataset is indeed much noisier than the validation dataset. If on the other hand, the loss_training is closer to loss_val, then the difference in loss computation mode is the reason for the difference in loss values.

In [ ]:
from pprint import pprint

cnn_train_eval = cnn_model.evaluate(
    train_ds,
    verbose = 2,
    return_dict = True,
 )

cnn_val_eval = cnn_model.evaluate(
    val_ds,
    verbose = 2,
    return_dict = True,
 )

cnn_test_eval = cnn_model.evaluate(
    test_ds,
    verbose = 2,
    return_dict = True,
 )

print("CNN Train Evaluation:")
pprint(cnn_train_eval, indent = 2)

print("CNN Validation Evaluation:")
pprint(cnn_val_eval, indent = 2)

print("CNN Test Evaluation:")
pprint(cnn_test_eval, indent = 2)

From the above `cnn_model.evaluate()` results, it seems the loss difference between training and validation still present - although not as pronounced as the loss evolution graph suggest. Hence, the difference that we see in the graph most likely is caused by how the two losses were computed during training.

## 🧠Food For Thought

The notebook reports both standard accuracy and top-3 accuracy. In the context of Indonesian food classification, explain when top-3 accuracy is a useful evaluation metric and when it might make the model look better than it really is. In your answer, consider how the model may be used in a real application.

## Model 2: VGGNet-Based Architecture

In this second model, we will use a VGGNet-based architecture. VGGNet is a convolutional neural network architecture that was introduced in the paper "Very Deep Convolutional Networks for Large-Scale Image Recognition" by Karen Simonyan and Andrew Zisserman.

The architecture consists of multiple convolutional layers followed by max-pooling layers, and then fully connected layers for classification. VGGNet has been shown to achieve state-of-the-art performance on various image classification tasks.

To use VGGNet as the basis for our Model02, we need to do `fine-tuning` a trained VGGNet model suitable for our `indonesian_food` image classification task.

## 🧠Food For Thought

The notebook compares a custom CNN with a VGG16-based transfer learning model pretrained on ImageNet. Explain why the pretrained model might perform better on this dataset, but also describe situations where the simpler custom CNN could still be the better option. In your answer, consider dataset size, computational cost, and the similarity between ImageNet features and Indonesian food images.

In [ ]:
from typing import Tuple

import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# We need to modify the image pre-processing
# from `rc_image_np = rc_image_np.astype(np.float32) / 255.0` --> which is used in our custom CNN
# into
# ```Python
# rc_image_np = rc_image_np.astype(np.float32)
# rc_image_np = tf.keras.applications.vgg16.preprocess_input(rc_image_np)
# ```

def load_and_preprocess_image_for_vgg(
        image_path_tensor: tf.Tensor
) -> tf.Tensor:

    image_path: str = image_path_tensor.numpy().decode("utf-8")

    with Image.open(image_path) as img:
        rc_image_np: np.ndarray = resize_and_crop_image(img = img)

    rc_image_np = rc_image_np.astype(np.float32)
    rc_image_np = tf.keras.applications.vgg16.preprocess_input(rc_image_np)

    rc_image_tf: tf.Tensor = tf.convert_to_tensor(rc_image_np)

    return rc_image_tf

In [ ]:
def tf_preprocess_wrapper_for_vgg(
        image_path: tf.Tensor
        , label: tf.Tensor
) -> Tuple[tf.Tensor, tf.Tensor]:

    image: tf.Tensor = tf.py_function(
        func = load_and_preprocess_image_for_vgg
        , inp = [image_path]
        , Tout = tf.float32
    )

    image.set_shape((224,224,3))

    return image, label

In [ ]:
def build_dataset_from_labeled_images_for_vgg(
        labeled_images: ListLabels
        , batch_size: int = 32
        , is_training: bool = False
) -> tf.data.Dataset:

    image_paths, labels = prepare_records(labeled_images = labeled_images)

    dst: tf.data.Dataset = tf.data.Dataset.from_tensor_slices(
        tensors = (image_paths, labels)
    )

    if is_training:
        dst = dst.shuffle(buffer_size = len(image_paths))

    dst = dst.map(tf_preprocess_wrapper_for_vgg, num_parallel_calls = tf.data.AUTOTUNE)
    dst = dst.batch(batch_size = batch_size)
    dst = dst.prefetch(buffer_size = tf.data.AUTOTUNE)

    return dst

In [ ]:
import tensorflow as tf

BATCH_SIZE: int = 32
EPOCHS: int = 30

train_ds: tf.data.Dataset = build_dataset_from_labeled_images_for_vgg(
    labeled_images = labeled_train_images
    , batch_size = BATCH_SIZE
    , is_training = True
)

val_ds: tf.data.Dataset = build_dataset_from_labeled_images_for_vgg(
    labeled_images = labeled_val_images
    , batch_size = BATCH_SIZE
    , is_training = False
)

test_ds: tf.data.Dataset = build_dataset_from_labeled_images_for_vgg(
    labeled_images = labeled_test_images
    , batch_size = BATCH_SIZE
    , is_training = False
)

In [ ]:
from typing import Tuple
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_vgg16_model(
        input_shape: Tuple[int, int, int] = (224, 224, 3)
        , num_classes: int = len(categories_name)
        , trainable_backbone: bool = False
) -> Model:

    inputs: tf.Tensor = layers.Input(shape = input_shape)

    # Here, we are using VGGNet16 as the basis of our CNN
    vgg_base: Model = tf.keras.applications.VGG16(
        include_top = False
        , weights = "imagenet"
        , input_tensor = inputs
    )

    vgg_base.trainable = trainable_backbone

    # Attaching the VGGNet Model to MLP like we did earlier
    x: tf.Tensor = vgg_base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation = "relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation = "relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs: tf.Tensor = layers.Dense(num_classes, activation = "softmax")(x)

    cnn_model: Model = Model(
        inputs = inputs
        , outputs = outputs
    )

    return cnn_model

In [ ]:
vgg_model: Model = build_vgg16_model()

vgg_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4)
    , loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = False)
    , metrics = [
        "accuracy"
        , tf.keras.metrics.SparseTopKCategoricalAccuracy(k = 3, name = "top_3_accuracy")
    ]
)

In [ ]:
callbacks: List[tf.keras.callbacks.Callback] = [
    tf.keras.callbacks.EarlyStopping(
        monitor = "val_loss"
        , patience = 5
        , restore_best_weights = True
    )
    , tf.keras.callbacks.ReduceLROnPlateau(
        monitor = "val_loss"
        , factor = 0.5
        , patience = 2
        , min_lr = 1e-6
    )
    , tf.keras.callbacks.ModelCheckpoint(
        filepath = str(model_path / "vgg_indonesian_food_model.keras")
        , monitor = "val_loss"
        , save_best_only = True
    )
]

In [ ]:
vgg_history: tf.keras.callbacks.History = vgg_model.fit(
    train_ds
    , validation_data = val_ds
    , epochs = EPOCHS
    , callbacks = callbacks
)

In [ ]:
plot_loss_history(history = vgg_history)

In [ ]:
from pprint import pprint

vgg_train_eval = vgg_model.evaluate(
    train_ds,
    verbose = 2,
    return_dict = True,
 )

vgg_val_eval = vgg_model.evaluate(
    val_ds,
    verbose = 2,
    return_dict = True,
 )

vgg_test_eval = vgg_model.evaluate(
    test_ds,
    verbose = 2,
    return_dict = True,
 )

print("VGG Train Evaluation:")
pprint(vgg_train_eval, indent = 2)

print("VGG Validation Evaluation:")
pprint(vgg_val_eval, indent = 2)

print("VGG Test Evaluation:")
pprint(vgg_test_eval, indent = 2)

## Final Comparison

The final comparison should focus on validation and test performance, not only training metrics. In this assignment, the most relevant metrics are `accuracy`, `top_3_accuracy`, and the stability of the validation loss curve.

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "model": "custom_cnn",
            "val_loss": cnn_val_eval.get("loss"),
            "val_accuracy": cnn_val_eval.get("accuracy"),
            "val_top_3_accuracy": cnn_val_eval.get("top_3_accuracy"),
            "test_loss": cnn_test_eval.get("loss"),
            "test_accuracy": cnn_test_eval.get("accuracy"),
            "test_top_3_accuracy": cnn_test_eval.get("top_3_accuracy"),
        },
        {
            "model": "vgg16_transfer_learning",
            "val_loss": vgg_val_eval.get("loss"),
            "val_accuracy": vgg_val_eval.get("accuracy"),
            "val_top_3_accuracy": vgg_val_eval.get("top_3_accuracy"),
            "test_loss": vgg_test_eval.get("loss"),
            "test_accuracy": vgg_test_eval.get("accuracy"),
            "test_top_3_accuracy": vgg_test_eval.get("top_3_accuracy"),
        },
    ]
)

comparison_df.sort_values(by = ["test_accuracy", "test_top_3_accuracy"], ascending = False)